In [1]:
import pandas as pd
import numpy as npy
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path

In [2]:
from pathlib import Path

import numpy as np
import pandas as pd


folder = Path(
    r"C:\Users\arhan\PycharmProjects\inflationstudymirror\Datas\Markets\Gurmar"
)

daily_dataframes = []
failed_files = []

for file in sorted(folder.glob("gurmar_prices_*.csv")):
    try:
        # sep=None attempts to detect comma/semicolon automatically.
        # utf-8-sig removes the invisible BOM character.
        df = pd.read_csv(
            file,
            encoding="utf-8-sig",
            sep=None,
            engine="python"
        )

        # Standardize column names.
        df.columns = (
            df.columns
            .str.replace("\ufeff", "", regex=False)
            .str.strip()
            .str.lower()
            .str.replace("_", "-", regex=False)
            .str.replace(" ", "-", regex=False)
        )

        # Convert known column variants into one naming standard.
        df = df.rename(
            columns={
                "product-name": "product_name",
                "product-price": "price",
                "product-id": "product_id",
                "kategori": "category",
                "tarih": "source_date",
                "eski-fiyat": "old_price"
            }
        )

        required_columns = {"product_name", "price"}

        if not required_columns.issubset(df.columns):
            missing = required_columns - set(df.columns)

            failed_files.append(
                {
                    "file": file.name,
                    "reason": f"Missing columns: {missing}"
                }
            )
            continue

        # Read the date from the filename.
        date_text = file.stem.replace("gurmar_prices_", "")
        df["date"] = pd.to_datetime(date_text, format="%Y-%m-%d")

        # Clean product names.
        df["product_name"] = (
            df["product_name"]
            .astype("string")
            .str.strip()
        )

        # Clean prices.
        df["price"] = (
            df["price"]
            .astype("string")
            .str.strip()
            .str.replace("₺", "", regex=False)
            .str.replace("TL", "", regex=False)
            .str.replace(" ", "", regex=False)
        )

        # Handles Turkish-style decimal values such as 1.249,90.
        contains_comma = df["price"].str.contains(",", na=False)

        df.loc[contains_comma, "price"] = (
            df.loc[contains_comma, "price"]
            .str.replace(".", "", regex=False)
            .str.replace(",", ".", regex=False)
        )

        df["price"] = pd.to_numeric(df["price"], errors="coerce")

        # Remove unusable rows.
        df = df.dropna(subset=["product_name", "price"])

        # Avoid duplicate product observations within the same daily file.
        duplicate_key = (
            ["product_id"]
            if "product_id" in df.columns
            else ["product_name"]
        )

        df = df.drop_duplicates(
            subset=duplicate_key,
            keep="last"
        )

        daily_dataframes.append(df)

        print(f"OK: {file.name} — {len(df)} rows")

    except Exception as error:
        failed_files.append(
            {
                "file": file.name,
                "reason": str(error)
            }
        )

        print(f"ERROR: {file.name} — {error}")


if not daily_dataframes:
    raise ValueError("No usable CSV files were found.")

# Stack files vertically.
data = pd.concat(
    daily_dataframes,
    ignore_index=True,
    sort=False
)

# Sort each product's history chronologically.
identity_column = (
    "product_id"
    if "product_id" in data.columns
    else "product_name"
)

data = data.sort_values(
    [identity_column, "date"]
).reset_index(drop=True)

OK: gurmar_prices_2026-02-21.csv — 4888 rows
OK: gurmar_prices_2026-02-22.csv — 4793 rows
OK: gurmar_prices_2026-02-24.csv — 4648 rows
OK: gurmar_prices_2026-02-26.csv — 4968 rows
OK: gurmar_prices_2026-02-27.csv — 5036 rows
OK: gurmar_prices_2026-02-28.csv — 5125 rows
OK: gurmar_prices_2026-03-01.csv — 5125 rows
OK: gurmar_prices_2026-03-02.csv — 4900 rows
OK: gurmar_prices_2026-03-03.csv — 3954 rows
OK: gurmar_prices_2026-03-04.csv — 5011 rows
OK: gurmar_prices_2026-03-05.csv — 4971 rows
OK: gurmar_prices_2026-03-06.csv — 5012 rows
OK: gurmar_prices_2026-03-08.csv — 5004 rows
OK: gurmar_prices_2026-03-09.csv — 4849 rows
OK: gurmar_prices_2026-03-10.csv — 4917 rows
OK: gurmar_prices_2026-03-11.csv — 4951 rows
OK: gurmar_prices_2026-03-12.csv — 5034 rows
OK: gurmar_prices_2026-03-13.csv — 5092 rows
OK: gurmar_prices_2026-03-14.csv — 5102 rows
OK: gurmar_prices_2026-03-15.csv — 5104 rows
OK: gurmar_prices_2026-03-16.csv — 4994 rows
OK: gurmar_prices_2026-03-17.csv — 4057 rows
OK: gurmar

In [3]:
data["day_of_week"] = data["date"].dt.dayofweek
data["day_name"] = data["date"].dt.day_name()

data["day_of_week_sin"] = np.sin(
    2 * np.pi * data["day_of_week"] / 7
)

data["day_of_week_cos"] = np.cos(
    2 * np.pi * data["day_of_week"] / 7
)

In [4]:
data.head()

,product_name,price,date,source_date,category,old_price,url,product_id,day_of_week,day_name,day_of_week_sin,day_of_week_cos
0,Burn Enerji İçeceği 250 Ml,60.0,2026-05-07,NaN,İçecekler,NaN,NaN,168.0,3,Thursday,0.433884,-0.900969
1,Burn Enerji İçeceği 250 Ml,60.0,2026-05-08,NaN,İçecekler,NaN,NaN,168.0,4,Friday,-0.433884,-0.900969
2,Burn Enerji İçeceği 250 Ml,60.0,2026-05-09,NaN,İçecekler,NaN,NaN,168.0,5,Saturday,-0.974928,-0.222521
3,Burn Enerji İçeceği 250 Ml,60.0,2026-05-10,NaN,İçecekler,NaN,NaN,168.0,6,Sunday,-0.781831,0.623490
4,Burn Enerji İçeceği 250 Ml,60.0,2026-05-11,NaN,İçecekler,NaN,NaN,168.0,0,Monday,0.000000,1.000000


In [5]:
data["day_of_month"] = data["date"].dt.day
data["week_of_year"] = data["date"].dt.isocalendar().week.astype(int)
data["month"] = data["date"].dt.month
data["quarter"] = data["date"].dt.quarter
data["is_weekend"] = data["day_of_week"].isin([5, 6]).astype(int)

data["month_sin"] = np.sin(
    2 * np.pi * (data["month"] - 1) / 12
)

data["month_cos"] = np.cos(
    2 * np.pi * (data["month"] - 1) / 12
)

In [6]:
data["day_of_month"] = data["date"].dt.day
data["week_of_year"] = data["date"].dt.isocalendar().week.astype(int)
data["month"] = data["date"].dt.month
data["quarter"] = data["date"].dt.quarter
data["is_weekend"] = data["day_of_week"].isin([5, 6]).astype(int)

data["month_sin"] = np.sin(
    2 * np.pi * (data["month"] - 1) / 12
)

data["month_cos"] = np.cos(
    2 * np.pi * (data["month"] - 1) / 12
)

In [7]:
preferred_columns = [
    "product_id",
    "product_name",
    "category",
    "date",
    "price",
    "day_of_week",
    "day_name",
    "day_of_week_sin",
    "day_of_week_cos",
    "day_of_month",
    "week_of_year",
    "month",
    "month_sin",
    "month_cos",
    "quarter",
    "is_weekend"
]

preferred_columns = [
    column
    for column in preferred_columns
    if column in data.columns
]

remaining_columns = [
    column
    for column in data.columns
    if column not in preferred_columns
]

data = data[preferred_columns + remaining_columns]

In [8]:
output_file = folder / "gurmar_prices_long.csv"

data.to_csv(
    output_file,
    index=False,
    encoding="utf-8-sig"
)

print("\nCompleted")
print(f"Output: {output_file}")
print(f"Rows: {len(data):,}")
print(f"Products: {data[identity_column].nunique():,}")
print(f"Dates: {data['date'].nunique():,}")
print(f"Range: {data['date'].min()} — {data['date'].max()}")

if failed_files:
    print("\nFiles with errors:")
    for item in failed_files:
        print(f"- {item['file']}: {item['reason']}")


Completed
Output: C:\Users\arhan\PycharmProjects\inflationstudymirror\Datas\Markets\Gurmar\gurmar_prices_long.csv
Rows: 362,457
Products: 442
Dates: 158
Range: 2026-02-21 00:00:00 — 2026-08-02 00:00:00

Files with errors:
- gurmar_prices_long.csv: time data "long" doesn't match format "%Y-%m-%d". You might want to try:
    - passing `format` if your strings have a consistent format;
    - passing `format='ISO8601'` if your strings are all ISO8601 but not necessarily in exactly the same format;
    - passing `format='mixed'`, and the format will be inferred for each element individually. You might want to use `dayfirst` alongside this.


In [9]:
data.info()

<class 'pandas.DataFrame'>
RangeIndex: 362457 entries, 0 to 362456
Data columns (total 19 columns):
 #   Column           Non-Null Count   Dtype         
---  ------           --------------   -----         
 0   product_id       24200 non-null   float64       
 1   product_name     362457 non-null  string        
 2   category         332426 non-null  object        
 3   date             362457 non-null  datetime64[us]
 4   price            362457 non-null  Float64       
 5   day_of_week      362457 non-null  int32         
 6   day_name         362457 non-null  str           
 7   day_of_week_sin  362457 non-null  float64       
 8   day_of_week_cos  362457 non-null  float64       
 9   day_of_month     362457 non-null  int32         
 10  week_of_year     362457 non-null  int64         
 11  month            362457 non-null  int32         
 12  month_sin        362457 non-null  float64       
 13  month_cos        362457 non-null  float64       
 14  quarter          362457 non-nul

In [10]:
data.head()

,product_id,product_name,category,date,price,day_of_week,day_name,day_of_week_sin,day_of_week_cos,day_of_month,week_of_year,month,month_sin,month_cos,quarter,is_weekend,source_date,old_price,url
0,168.0,Burn Enerji İçeceği 250 Ml,İçecekler,2026-05-07,60.0,3,Thursday,0.433884,-0.900969,7,19,5,0.866025,-0.5,2,0,NaN,NaN,NaN
1,168.0,Burn Enerji İçeceği 250 Ml,İçecekler,2026-05-08,60.0,4,Friday,-0.433884,-0.900969,8,19,5,0.866025,-0.5,2,0,NaN,NaN,NaN
2,168.0,Burn Enerji İçeceği 250 Ml,İçecekler,2026-05-09,60.0,5,Saturday,-0.974928,-0.222521,9,19,5,0.866025,-0.5,2,1,NaN,NaN,NaN
3,168.0,Burn Enerji İçeceği 250 Ml,İçecekler,2026-05-10,60.0,6,Sunday,-0.781831,0.623490,10,19,5,0.866025,-0.5,2,1,NaN,NaN,NaN
4,168.0,Burn Enerji İçeceği 250 Ml,İçecekler,2026-05-11,60.0,0,Monday,0.000000,1.000000,11,20,5,0.866025,-0.5,2,0,NaN,NaN,NaN


In [11]:
data.describe()

,product_id,date,price,day_of_week,day_of_week_sin,day_of_week_cos,day_of_month,week_of_year,month,month_sin,month_cos,quarter,is_weekend
count,24200.000000,362457,362457.0,362457.000000,362457.000000,362457.000000,362457.000000,362457.000000,362457.000000,362457.000000,3.624570e+05,362457.000000,362457.000000
mean,7276.451198,2026-04-04 07:29:06.172373,166.456409,3.092990,-0.036984,0.010065,15.735185,14.316984,3.628419,0.857642,1.914320e-01,1.537286,0.306668
min,168.000000,2026-02-21 00:00:00,0.5,0.000000,-0.974928,-0.900969,1.000000,8.000000,2.000000,-0.500000,-1.000000e+00,1.000000,0.000000
25%,4125.000000,2026-03-14 00:00:00,56.75,1.000000,-0.781831,-0.900969,8.000000,11.000000,3.000000,0.866025,6.123234e-17,1.000000,0.000000
50%,7394.000000,2026-04-01 00:00:00,109.95,3.000000,0.000000,-0.222521,16.000000,14.000000,4.000000,0.866025,6.123234e-17,2.000000,0.000000
75%,10860.000000,2026-04-20 00:00:00,214.9,5.000000,0.781831,0.623490,24.000000,17.000000,4.000000,1.000000,5.000000e-01,2.000000,1.000000
max,13422.000000,2026-08-02 00:00:00,2890.0,6.000000,0.974928,1.000000,31.000000,31.000000,8.000000,1.000000,8.660254e-01,3.000000,1.000000
std,3739.549975,NaN,187.889733,2.028525,0.707592,0.705583,9.066812,4.175359,0.997897,0.203284,4.318342e-01,0.546521,0.461111


In [12]:
data.isna().sum().sort_values(ascending=False)

old_price          361819
source_date        357809
url                357809
product_id         338257
category            30031
product_name            0
date                    0
day_of_week_sin         0
price                   0
day_of_week             0
day_name                0
week_of_year            0
day_of_month            0
day_of_week_cos         0
month                   0
quarter                 0
month_cos               0
month_sin               0
is_weekend              0
dtype: int64

In [13]:
data["category"] = (
    data.groupby("product_name")["category"]
        .transform(lambda x: x.ffill().bfill())
)

In [14]:
data["product_id"] = (
    data.groupby("product_name")["product_id"]
        .transform(lambda x: x.ffill().bfill())
)

In [15]:
data = data.drop(
    columns=[
        "url",
        "source_date",
        "old_price"
    ],
    errors="ignore"
)

In [16]:
data.isna().sum().sort_values(ascending=False)

product_id         316814
category             1442
product_name            0
date                    0
price                   0
day_of_week             0
day_name                0
day_of_week_sin         0
day_of_week_cos         0
day_of_month            0
week_of_year            0
month                   0
month_sin               0
month_cos               0
quarter                 0
is_weekend              0
dtype: int64

In [17]:
data["series_id"] = np.where(
    data["product_id"].notna(),
    "id_" + data["product_id"].astype(str),
    "name_" + data["product_name"].astype(str)
)

data[["product_id", "product_name", "series_id"]].head()

,product_id,product_name,series_id
0,168.0,Burn Enerji İçeceği 250 Ml,id_168.0
1,168.0,Burn Enerji İçeceği 250 Ml,id_168.0
2,168.0,Burn Enerji İçeceği 250 Ml,id_168.0
3,168.0,Burn Enerji İçeceği 250 Ml,id_168.0
4,168.0,Burn Enerji İçeceği 250 Ml,id_168.0


In [18]:
data = data.sort_values(
    ["series_id", "date"]
).reset_index(drop=True)

data[["series_id", "date", "price"]].head(20)

,series_id,date,price
0,id_10066.0,2026-02-21,36.95
1,id_10066.0,2026-02-22,36.95
2,id_10066.0,2026-02-24,36.95
3,id_10066.0,2026-02-26,36.95
4,id_10066.0,2026-02-27,36.95
5,id_10066.0,2026-02-28,36.95
6,id_10066.0,2026-03-01,36.95
7,id_10066.0,2026-03-02,36.95
8,id_10066.0,2026-03-04,36.95
9,id_10066.0,2026-03-05,36.95


In [19]:
grouped_price = data.groupby("series_id")["price"]

data["price_lag_1"] = grouped_price.shift(1)
data["price_lag_7"] = grouped_price.shift(7)
data["price_lag_14"] = grouped_price.shift(14)

data["price_mean_7"] = grouped_price.transform(
    lambda x: x.shift(1).rolling(7).mean()
)

data["price_mean_14"] = grouped_price.transform(
    lambda x: x.shift(1).rolling(14).mean()
)

data["price_std_7"] = grouped_price.transform(
    lambda x: x.shift(1).rolling(7).std()
)

In [20]:
data[
    [
        "series_id",
        "date",
        "price",
        "price_lag_1",
        "price_lag_7",
        "price_lag_14",
        "price_mean_7",
        "price_mean_14",
        "price_std_7"
    ]
].head(25)

,series_id,date,price,price_lag_1,price_lag_7,price_lag_14,price_mean_7,price_mean_14,price_std_7
0,id_10066.0,2026-02-21,36.95,<NA>,<NA>,<NA>,NaN,NaN,NaN
1,id_10066.0,2026-02-22,36.95,36.95,<NA>,<NA>,NaN,NaN,NaN
2,id_10066.0,2026-02-24,36.95,36.95,<NA>,<NA>,NaN,NaN,NaN
3,id_10066.0,2026-02-26,36.95,36.95,<NA>,<NA>,NaN,NaN,NaN
4,id_10066.0,2026-02-27,36.95,36.95,<NA>,<NA>,NaN,NaN,NaN
5,id_10066.0,2026-02-28,36.95,36.95,<NA>,<NA>,NaN,NaN,NaN
6,id_10066.0,2026-03-01,36.95,36.95,<NA>,<NA>,NaN,NaN,NaN
7,id_10066.0,2026-03-02,36.95,36.95,36.95,<NA>,36.95,NaN,0.0
8,id_10066.0,2026-03-04,36.95,36.95,36.95,<NA>,36.95,NaN,0.0
9,id_10066.0,2026-03-05,36.95,36.95,36.95,<NA>,36.95,NaN,0.0


In [21]:
feature_columns = [
    "day_of_week_sin",
    "day_of_week_cos",
    "month_sin",
    "month_cos",
    "is_weekend",
    "price_lag_1",
    "price_lag_7",
    "price_lag_14",
    "price_mean_7",
    "price_mean_14",
    "price_std_7"
]

In [22]:
split_date = pd.Timestamp("2026-07-01")

train_df = data[data["date"] < split_date].copy()
test_df = data[data["date"] >= split_date].copy()

print(
    "Train tarih aralığı:",
    train_df["date"].min(),
    "→",
    train_df["date"].max()
)

print(
    "Test tarih aralığı:",
    test_df["date"].min(),
    "→",
    test_df["date"].max()
)

Train tarih aralığı: 2026-02-21 00:00:00 → 2026-06-30 00:00:00
Test tarih aralığı: 2026-07-01 00:00:00 → 2026-08-02 00:00:00


In [23]:
train_df = train_df.dropna(
    subset=feature_columns + ["price"]
).copy()

test_df = test_df.dropna(
    subset=feature_columns + ["price"]
).copy()

print("Train satır sayısı:", len(train_df))
print("Test satır sayısı:", len(test_df))

Train satır sayısı: 270188
Test satır sayısı: 8792


In [24]:
X_train = train_df[feature_columns]
y_train = train_df["price"]

X_test = test_df[feature_columns]
y_test = test_df["price"]

print("X_train:", X_train.shape)
print("X_test:", X_test.shape)

X_train: (270188, 11)
X_test: (8792, 11)


In [25]:
from sklearn.ensemble import HistGradientBoostingRegressor

model = HistGradientBoostingRegressor(
    learning_rate=0.05,
    max_iter=200,
    max_leaf_nodes=31,
    random_state=42
)

model.fit(X_train, y_train)

,"learning_rate learning_rate: float, default=0.1The learning rate, also known as *shrinkage*. This is used as amultiplicative factor for the leaves values. Use ``1`` for noshrinkage.",0.05
,"max_iter max_iter: int, default=100The maximum number of iterations of the boosting process, i.e. themaximum number of trees.",200
,"random_state random_state: int, RandomState instance or None, default=NonePseudo-random number generator to control the subsampling in thebinning process, and the train/validation data split if early stoppingis enabled.Pass an int for reproducible output across multiple function calls.See :term:`Glossary <random_state>`.",42
,"loss loss: {'squared_error', 'absolute_error', 'gamma', 'poisson', 'quantile'}, default='squared_error'The loss function to use in the boosting process. Note that the""squared error"", ""gamma"" and ""poisson"" losses actually implement""half least squares loss"", ""half gamma deviance"" and ""half poissondeviance"" to simplify the computation of the gradient. Furthermore,""gamma"" and ""poisson"" losses internally use a log-link, ""gamma""requires ``y > 0`` and ""poisson"" requires ``y >= 0``.""quantile"" uses the pinball loss... versionchanged:: 0.23 Added option 'poisson'... versionchanged:: 1.1 Added option 'quantile'... versionchanged:: 1.3 Added option 'gamma'.",'squared_error'
,"quantile quantile: float, default=NoneIf loss is ""quantile"", this parameter specifies which quantile to be estimatedand must be between 0 and 1.",None
,"max_leaf_nodes max_leaf_nodes: int or None, default=31The maximum number of leaves for each tree. Must be strictly greaterthan 1. If None, there is no maximum limit.",31
,"max_depth max_depth: int or None, default=NoneThe maximum depth of each tree. The depth of a tree is the number ofedges to go from the root to the deepest leaf.Depth isn't constrained by default.",None
,"min_samples_leaf min_samples_leaf: int, default=20The minimum number of samples per leaf. For small datasets with lessthan a few hundred samples, it is recommended to lower this valuesince only very shallow trees would be built.",20
,"l2_regularization l2_regularization: float, default=0The L2 regularization parameter penalizing leaves with small hessians.Use ``0`` for no regularization (default).",0.0
,"max_features max_features: float, default=1.0Proportion of randomly chosen features in each and every node split.This is a form of regularization, smaller values make the trees weakerlearners and might prevent overfitting.If interaction constraints from `interaction_cst` are present, only allowedfeatures are taken into account for the subsampling... versionadded:: 1.4",1.0
,"max_bins max_bins: int, default=255The maximum number of bins to use for non-missing values. Beforetraining, each feature of the input array `X` is binned intointeger-valued bins, which allows for a much faster training stage.Features with a small number of unique values may use less than``max_bins`` bins. In addition to the ``max_bins`` bins, one more binis always reserved for missing values. Must be no larger than 255.",255


In [26]:
test_df["prediction"] = model.predict(X_test)

test_df[
    ["series_id", "date", "price", "prediction"]
].head(20)

,series_id,date,price,prediction
56,id_10066.0,2026-07-15,39.95,37.375639
57,id_10066.0,2026-07-16,39.95,40.263530
122,id_10068.0,2026-07-03,37.95,38.059182
123,id_10068.0,2026-07-04,37.95,38.016597
124,id_10068.0,2026-07-05,37.95,38.026722
125,id_10068.0,2026-07-06,37.95,38.083469
126,id_10068.0,2026-07-07,37.95,38.097298
127,id_10068.0,2026-07-08,37.95,38.101473
128,id_10068.0,2026-07-09,37.95,37.991142
129,id_10068.0,2026-07-10,37.95,38.059182


In [27]:
from sklearn.metrics import (
    mean_absolute_error,
    mean_squared_error,
    r2_score
)

mae = mean_absolute_error(
    test_df["price"],
    test_df["prediction"]
)

rmse = mean_squared_error(
    test_df["price"],
    test_df["prediction"]
) ** 0.5

r2 = r2_score(
    test_df["price"],
    test_df["prediction"]
)

print(f"MAE  : {mae:.4f}")
print(f"RMSE : {rmse:.4f}")
print(f"R²   : {r2:.4f}")

MAE  : 7.1749
RMSE : 32.4922
R²   : 0.9819


# 🗺️ Gurmar Fiyat Tahmin Pipeline Haritası

Bu notebook'ta amaç, günlük market fiyat dosyalarını kullanarak **tek bir global model ile bütün ürünlerin fiyatlarını tahmin etmek**.

Model her ürün için ayrı ayrı eğitilmez. Bunun yerine bütün ürünlerin geçmiş fiyat örüntülerinden öğrenen tek bir model oluşturulur.

---

## Genel Akış

```text
Günlük CSV dosyaları
        │
        ▼
CSV dosyalarını standartlaştırma
        │
        ▼
Long format veri seti oluşturma
        │
        ▼
Veri temizleme
        │
        ▼
Takvim özellikleri oluşturma
        │
        ▼
Her ürün için zaman serisi kimliği oluşturma
        │
        ▼
Lag ve rolling özellikleri oluşturma
        │
        ▼
Tarihe göre Train / Test ayrımı
        │
        ▼
Global forecasting modelini eğitme
        │
        ▼
Bütün ürünler için tahmin üretme
        │
        ▼
MAE, RMSE, MAPE ve R² ile değerlendirme
        │
        ▼
Basit Lag-1 tahminiyle karşılaştırma

In [28]:
evaluation_df = test_df.copy()

evaluation_df["baseline_last_price"] = evaluation_df["price_lag_1"]
evaluation_df["baseline_last_week"] = evaluation_df["price_lag_7"]
evaluation_df["baseline_mean_7"] = evaluation_df["price_mean_7"]

In [29]:
from sklearn.metrics import (
    mean_absolute_error,
    mean_squared_error,
    r2_score
)

def regression_metrics(y_true, y_pred):
    valid = y_true.notna() & y_pred.notna()

    y_true = y_true[valid]
    y_pred = y_pred[valid]

    return {
        "MAE": mean_absolute_error(y_true, y_pred),
        "RMSE": mean_squared_error(y_true, y_pred) ** 0.5,
        "R2": r2_score(y_true, y_pred)
    }


results = []

predictions = {
    "HistGradientBoosting": evaluation_df["prediction"],
    "Last price": evaluation_df["baseline_last_price"],
    "Last week": evaluation_df["baseline_last_week"],
    "Rolling mean 7": evaluation_df["baseline_mean_7"]
}

for model_name, prediction in predictions.items():
    metrics = regression_metrics(
        evaluation_df["price"],
        prediction
    )

    results.append({
        "model": model_name,
        **metrics
    })

comparison_df = pd.DataFrame(results).sort_values("MAE")
comparison_df

,model,MAE,RMSE,R2
1,Last price,2.668074,18.666267,0.994025
0,HistGradientBoosting,7.174877,32.492201,0.981897
3,Rolling mean 7,8.079340,28.730646,0.985846
2,Last week,12.885417,42.458925,0.969087


In [30]:
test_df["actual_price_change"] = (
    test_df["price"] - test_df["price_lag_1"]
)

test_df["price_changed"] = (
    test_df["actual_price_change"].abs() > 0.01
)

In [31]:
print(
    test_df["price_changed"].value_counts()
)

print(
    test_df["price_changed"].value_counts(normalize=True)
)

price_changed
False    8155
True      637
Name: count, dtype: Int64
price_changed
False    0.927548
True     0.072452
Name: proportion, dtype: Float64


In [32]:
changed_test = test_df[
    test_df["price_changed"]
].copy()

print("Fiyat değişen test satırı:", len(changed_test))

Fiyat değişen test satırı: 637


In [33]:
changed_model_mae = mean_absolute_error(
    changed_test["price"],
    changed_test["prediction"]
)

changed_baseline_mae = mean_absolute_error(
    changed_test["price"],
    changed_test["price_lag_1"]
)

print("Değişim günleri model MAE:", changed_model_mae)
print("Değişim günleri Lag-1 MAE:", changed_baseline_mae)

Değişim günleri model MAE: 39.69194493294745
Değişim günleri Lag-1 MAE: 36.82529042386185


In [34]:
test_df["predicted_price_change"] = (
    test_df["prediction"] - test_df["price_lag_1"]
)

test_df["actual_direction"] = np.sign(
    test_df["actual_price_change"]
)

test_df["predicted_direction"] = np.sign(
    test_df["predicted_price_change"]
)

In [35]:
changed_test = test_df[
    test_df["actual_direction"] != 0
].copy()

direction_accuracy = (
    changed_test["actual_direction"]
    == changed_test["predicted_direction"]
).mean()

print(f"Değişim yönü doğruluğu: %{direction_accuracy * 100:.2f}")

Değişim yönü doğruluğu: %56.20


In [36]:
pd.crosstab(
    changed_test["actual_direction"],
    changed_test["predicted_direction"],
    rownames=["Gerçek"],
    colnames=["Tahmin"]
)

Tahmin,-1.0,1.0
Gerçek,,
-1.0,123,189
1.0,90,235


Yukardaki model dünki fiyat değişmeyecek demekten daha kötü performans gösteriyor. Bu yüzden daha düzgün bir train test split yöntemine ihtiyaç var

# 🔄 Walk-Forward Validation

## Neden Walk-Forward Validation?

Time series problemlerinde klasik train/test ayrımı her zaman yeterli değildir.

Tek bir test dönemi;

- kolay bir döneme,
- zor bir döneme,
- fiyatların çok değişmediği bir döneme

denk gelmiş olabilir.

Bu durumda model performansı gerçeği yansıtmayabilir.

---

## Amaç

Modeli yalnızca tek bir tarih aralığında değil,
birden fazla zaman aralığında test ederek
daha güvenilir bir performans tahmini elde etmek.

---

## Mantık

Her iterasyonda eğitim kümesi biraz daha büyür.

```text
Fold 1

Train
[================]

Test
                [====]


Fold 2

Train
[====================]

Test
                    [====]


Fold 3

Train
[========================]

Test
                        [====]
```

Model her fold'da yeniden eğitilir.

Her fold sonunda aşağıdaki metrikler hesaplanır:

- MAE
- RMSE
- MAPE
- R²

ve ayrıca

- Last Price Baseline

ile karşılaştırılır.

---

## Avantajları

✔ Gerçek hayattaki kullanım senaryosunu simüle eder.

✔ Tek bir split'e bağlı kalmaz.

✔ Modelin zaman içinde tutarlı çalışıp çalışmadığını gösterir.

✔ Overfitting'i anlamayı kolaylaştırır.

✔ Akademik çalışmalarda ve endüstride önerilen değerlendirme yöntemlerinden biridir.

---

## Beklenen Pipeline

```text
Fold 1
Train ─────────► Test

↓

Model Eğit

↓

Tahmin

↓

Metrikleri Hesapla

↓

Fold 2

↓

Fold 3

↓

...

↓

Fold Sonuçlarını Ortalama Al
```

---

## Başarı Kriteri

Her fold için

Model MAE

ile

Last Price Baseline MAE

karşılaştırılır.

```text
Model MAE
        <
Last Price MAE
```

ise model,
basit "yarın = bugün" tahmininden daha başarılıdır.

Aksi durumda,
oluşturulan model henüz baseline'ı geçememiş demektir.

---

## Nihai Amaç

Walk-Forward Validation sonunda tek bir sayı yerine

| Fold | Model MAE | Baseline MAE |
|------|-----------|--------------|
| 1 | ... | ... |
| 2 | ... | ... |
| 3 | ... | ... |

elde edilir.

Böylece modelin farklı zaman aralıklarında
kararlı çalışıp çalışmadığı analiz edilir.

In [37]:
from sklearn.base import clone
from sklearn.ensemble import HistGradientBoostingRegressor
from sklearn.metrics import (
    mean_absolute_error,
    mean_squared_error,
    mean_absolute_percentage_error,
    r2_score
)

from xgboost import XGBRegressor
from lightgbm import LGBMRegressor
from catboost import CatBoostRegressor

In [38]:
walk_forward_folds = [
    {
        "fold": "Fold 1",
        "train_end": pd.Timestamp("2026-06-01"),
        "test_end": pd.Timestamp("2026-06-15")
    },
    {
        "fold": "Fold 2",
        "train_end": pd.Timestamp("2026-06-15"),
        "test_end": pd.Timestamp("2026-07-01")
    },
    {
        "fold": "Fold 3",
        "train_end": pd.Timestamp("2026-07-01"),
        "test_end": pd.Timestamp("2026-07-18")
    }
]

pd.DataFrame(walk_forward_folds)

,fold,train_end,test_end
0,Fold 1,2026-06-01,2026-06-15
1,Fold 2,2026-06-15,2026-07-01
2,Fold 3,2026-07-01,2026-07-18


In [39]:
def calculate_forecasting_metrics(y_true, y_pred):
    y_true = pd.Series(y_true).reset_index(drop=True)
    y_pred = pd.Series(y_pred).reset_index(drop=True)

    valid_mask = y_true.notna() & y_pred.notna()

    y_true = y_true[valid_mask]
    y_pred = y_pred[valid_mask]

    if len(y_true) == 0:
        return {
            "MAE": np.nan,
            "RMSE": np.nan,
            "MAPE": np.nan,
            "R2": np.nan
        }

    return {
        "MAE": mean_absolute_error(y_true, y_pred),
        "RMSE": mean_squared_error(y_true, y_pred) ** 0.5,
        "MAPE": mean_absolute_percentage_error(y_true, y_pred) * 100,
        "R2": r2_score(y_true, y_pred)
    }

In [40]:
def run_walk_forward_validation(
    data,
    model,
    model_name,
    feature_columns,
    folds
):
    fold_results = []
    fold_predictions = []

    required_columns = feature_columns + [
        "price",
        "price_lag_1",
        "series_id",
        "date"
    ]

    for fold_info in folds:
        train_end = fold_info["train_end"]
        test_end = fold_info["test_end"]
        fold_name = fold_info["fold"]

        fold_train = data[
            data["date"] < train_end
        ].dropna(
            subset=required_columns
        ).copy()

        fold_test = data[
            (data["date"] >= train_end)
            & (data["date"] < test_end)
        ].dropna(
            subset=required_columns
        ).copy()

        if fold_train.empty or fold_test.empty:
            print(f"{fold_name} atlandı: train veya test boş.")
            continue

        X_fold_train = fold_train[feature_columns]
        y_fold_train = fold_train["price"]

        X_fold_test = fold_test[feature_columns]
        y_fold_test = fold_test["price"]

        fold_model = clone(model)

        fold_model.fit(
            X_fold_train,
            y_fold_train
        )

        fold_test["prediction"] = fold_model.predict(
            X_fold_test
        )

        model_metrics = calculate_forecasting_metrics(
            fold_test["price"],
            fold_test["prediction"]
        )

        baseline_metrics = calculate_forecasting_metrics(
            fold_test["price"],
            fold_test["price_lag_1"]
        )

        fold_test["actual_price_change"] = (
            fold_test["price"]
            - fold_test["price_lag_1"]
        )

        changed_rows = fold_test[
            fold_test["actual_price_change"].abs() > 0.01
        ].copy()

        if len(changed_rows) > 0:
            changed_model_mae = mean_absolute_error(
                changed_rows["price"],
                changed_rows["prediction"]
            )

            changed_baseline_mae = mean_absolute_error(
                changed_rows["price"],
                changed_rows["price_lag_1"]
            )
        else:
            changed_model_mae = np.nan
            changed_baseline_mae = np.nan

        fold_results.append({
            "model": model_name,
            "fold": fold_name,
            "train_end": train_end,
            "test_end": test_end,
            "train_rows": len(fold_train),
            "test_rows": len(fold_test),
            "changed_rows": len(changed_rows),

            "model_MAE": model_metrics["MAE"],
            "model_RMSE": model_metrics["RMSE"],
            "model_MAPE": model_metrics["MAPE"],
            "model_R2": model_metrics["R2"],

            "baseline_MAE": baseline_metrics["MAE"],
            "baseline_RMSE": baseline_metrics["RMSE"],
            "baseline_MAPE": baseline_metrics["MAPE"],
            "baseline_R2": baseline_metrics["R2"],

            "changed_model_MAE": changed_model_mae,
            "changed_baseline_MAE": changed_baseline_mae,

            "MAE_improvement": (
                baseline_metrics["MAE"]
                - model_metrics["MAE"]
            )
        })

        fold_test["fold"] = fold_name
        fold_test["model"] = model_name

        fold_predictions.append(
            fold_test[
                [
                    "model",
                    "fold",
                    "series_id",
                    "product_name",
                    "date",
                    "price",
                    "price_lag_1",
                    "prediction",
                    "actual_price_change"
                ]
            ]
        )

    results_df = pd.DataFrame(fold_results)

    if fold_predictions:
        predictions_df = pd.concat(
            fold_predictions,
            ignore_index=True
        )
    else:
        predictions_df = pd.DataFrame()

    return results_df, predictions_df

In [41]:
hist_model = HistGradientBoostingRegressor(
    learning_rate=0.05,
    max_iter=200,
    max_leaf_nodes=31,
    random_state=42
)

hist_model

,"learning_rate learning_rate: float, default=0.1The learning rate, also known as *shrinkage*. This is used as amultiplicative factor for the leaves values. Use ``1`` for noshrinkage.",0.05
,"max_iter max_iter: int, default=100The maximum number of iterations of the boosting process, i.e. themaximum number of trees.",200
,"random_state random_state: int, RandomState instance or None, default=NonePseudo-random number generator to control the subsampling in thebinning process, and the train/validation data split if early stoppingis enabled.Pass an int for reproducible output across multiple function calls.See :term:`Glossary <random_state>`.",42
,"loss loss: {'squared_error', 'absolute_error', 'gamma', 'poisson', 'quantile'}, default='squared_error'The loss function to use in the boosting process. Note that the""squared error"", ""gamma"" and ""poisson"" losses actually implement""half least squares loss"", ""half gamma deviance"" and ""half poissondeviance"" to simplify the computation of the gradient. Furthermore,""gamma"" and ""poisson"" losses internally use a log-link, ""gamma""requires ``y > 0`` and ""poisson"" requires ``y >= 0``.""quantile"" uses the pinball loss... versionchanged:: 0.23 Added option 'poisson'... versionchanged:: 1.1 Added option 'quantile'... versionchanged:: 1.3 Added option 'gamma'.",'squared_error'
,"quantile quantile: float, default=NoneIf loss is ""quantile"", this parameter specifies which quantile to be estimatedand must be between 0 and 1.",None
,"max_leaf_nodes max_leaf_nodes: int or None, default=31The maximum number of leaves for each tree. Must be strictly greaterthan 1. If None, there is no maximum limit.",31
,"max_depth max_depth: int or None, default=NoneThe maximum depth of each tree. The depth of a tree is the number ofedges to go from the root to the deepest leaf.Depth isn't constrained by default.",None
,"min_samples_leaf min_samples_leaf: int, default=20The minimum number of samples per leaf. For small datasets with lessthan a few hundred samples, it is recommended to lower this valuesince only very shallow trees would be built.",20
,"l2_regularization l2_regularization: float, default=0The L2 regularization parameter penalizing leaves with small hessians.Use ``0`` for no regularization (default).",0.0
,"max_features max_features: float, default=1.0Proportion of randomly chosen features in each and every node split.This is a form of regularization, smaller values make the trees weakerlearners and might prevent overfitting.If interaction constraints from `interaction_cst` are present, only allowedfeatures are taken into account for the subsampling... versionadded:: 1.4",1.0
,"max_bins max_bins: int, default=255The maximum number of bins to use for non-missing values. Beforetraining, each feature of the input array `X` is binned intointeger-valued bins, which allows for a much faster training stage.Features with a small number of unique values may use less than``max_bins`` bins. In addition to the ``max_bins`` bins, one more binis always reserved for missing values. Must be no larger than 255.",255


In [42]:
hist_wf_results, hist_wf_predictions = (
    run_walk_forward_validation(
        data=data,
        model=hist_model,
        model_name="HistGradientBoosting",
        feature_columns=feature_columns,
        folds=walk_forward_folds
    )
)

hist_wf_results

,model,fold,train_end,test_end,train_rows,test_rows,changed_rows,model_MAE,model_RMSE,model_MAPE,model_R2,baseline_MAE,baseline_RMSE,baseline_MAPE,baseline_R2,changed_model_MAE,changed_baseline_MAE,MAE_improvement
0,HistGradientBoosting,Fold 1,2026-06-01,2026-06-15,262190,3703,223,4.914097,24.039975,4.305553,0.988096,1.637618,12.116339,1.177737,0.996976,26.231601,27.193274,-3.276479
1,HistGradientBoosting,Fold 2,2026-06-15,2026-07-01,265893,4295,206,6.381608,30.036416,4.116500,0.984008,1.814137,13.895255,0.849934,0.996578,38.824640,37.823883,-4.567470
2,HistGradientBoosting,Fold 3,2026-07-01,2026-07-18,270188,4516,351,7.523046,35.645910,4.279386,0.976152,3.133459,20.536875,1.691849,0.992084,42.666853,40.315385,-4.389587


In [43]:
hist_wf_results[
    [
        "fold",
        "train_rows",
        "test_rows",
        "changed_rows",
        "model_MAE",
        "baseline_MAE",
        "MAE_improvement",
        "changed_model_MAE",
        "changed_baseline_MAE",
        "model_R2"
    ]
]

,fold,train_rows,test_rows,changed_rows,model_MAE,baseline_MAE,MAE_improvement,changed_model_MAE,changed_baseline_MAE,model_R2
0,Fold 1,262190,3703,223,4.914097,1.637618,-3.276479,26.231601,27.193274,0.988096
1,Fold 2,265893,4295,206,6.381608,1.814137,-4.567470,38.824640,37.823883,0.984008
2,Fold 3,270188,4516,351,7.523046,3.133459,-4.389587,42.666853,40.315385,0.976152


In [44]:
hist_wf_summary = pd.DataFrame({
    "metric": [
        "Model MAE",
        "Baseline MAE",
        "MAE Improvement",
        "Changed Model MAE",
        "Changed Baseline MAE",
        "Model RMSE",
        "Model MAPE",
        "Model R2"
    ],
    "mean": [
        hist_wf_results["model_MAE"].mean(),
        hist_wf_results["baseline_MAE"].mean(),
        hist_wf_results["MAE_improvement"].mean(),
        hist_wf_results["changed_model_MAE"].mean(),
        hist_wf_results["changed_baseline_MAE"].mean(),
        hist_wf_results["model_RMSE"].mean(),
        hist_wf_results["model_MAPE"].mean(),
        hist_wf_results["model_R2"].mean()
    ],
    "std": [
        hist_wf_results["model_MAE"].std(),
        hist_wf_results["baseline_MAE"].std(),
        hist_wf_results["MAE_improvement"].std(),
        hist_wf_results["changed_model_MAE"].std(),
        hist_wf_results["changed_baseline_MAE"].std(),
        hist_wf_results["model_RMSE"].std(),
        hist_wf_results["model_MAPE"].std(),
        hist_wf_results["model_R2"].std()
    ]
})

hist_wf_summary

,metric,mean,std
0,Model MAE,6.272917,1.307866
1,Baseline MAE,2.195071,0.817446
2,MAE Improvement,-4.077845,0.699680
3,Changed Model MAE,35.907698,8.597139
4,Changed Baseline MAE,35.110847,6.969066
5,Model RMSE,29.907434,5.804042
6,Model MAPE,4.233813,0.102435
7,Model R2,0.982752,0.006070


In [45]:
models = {
    "XGBoost": XGBRegressor(
        n_estimators=300,
        learning_rate=0.05,
        max_depth=6,
        subsample=0.8,
        colsample_bytree=0.8,
        objective="reg:squarederror",
        random_state=42,
        n_jobs=-1
    ),

    "LightGBM": LGBMRegressor(
        n_estimators=300,
        learning_rate=0.05,
        num_leaves=31,
        subsample=0.8,
        colsample_bytree=0.8,
        random_state=42,
        n_jobs=-1,
        verbosity=-1
    ),

    "CatBoost": CatBoostRegressor(
        iterations=300,
        learning_rate=0.05,
        depth=6,
        loss_function="RMSE",
        random_seed=42,
        verbose=False,
        allow_writing_files=False
    )
}

In [46]:
classic_split_results = []
classic_split_predictions = {}

for model_name, candidate_model in models.items():
    current_model = clone(candidate_model)

    current_model.fit(
        X_train,
        y_train
    )

    current_predictions = current_model.predict(
        X_test
    )

    metrics = calculate_forecasting_metrics(
        y_test,
        current_predictions
    )

    changed_mask = (
        test_df["price"] - test_df["price_lag_1"]
    ).abs() > 0.01

    changed_actual = test_df.loc[
        changed_mask,
        "price"
    ]

    changed_prediction = current_predictions[
        changed_mask.to_numpy()
    ]

    changed_model_mae = mean_absolute_error(
        changed_actual,
        changed_prediction
    )

    changed_baseline_mae = mean_absolute_error(
        changed_actual,
        test_df.loc[changed_mask, "price_lag_1"]
    )

    classic_split_results.append({
        "model": model_name,
        "MAE": metrics["MAE"],
        "RMSE": metrics["RMSE"],
        "MAPE": metrics["MAPE"],
        "R2": metrics["R2"],
        "changed_MAE": changed_model_mae,
        "changed_baseline_MAE": changed_baseline_mae
    })

    classic_split_predictions[model_name] = (
        current_predictions
    )

classic_split_results = pd.DataFrame(
    classic_split_results
).sort_values("MAE")

classic_split_results

,model,MAE,RMSE,MAPE,R2,changed_MAE,changed_baseline_MAE
1,LightGBM,4.814953,19.411334,3.506027,0.993539,35.737021,36.82529
2,CatBoost,6.706927,29.025122,4.879151,0.985554,36.221466,36.82529
0,XGBoost,7.672987,34.534578,3.851350,0.979549,40.431450,36.82529


In [47]:
baseline_classic_metrics = calculate_forecasting_metrics(
    y_test,
    test_df["price_lag_1"]
)

baseline_changed_mask = (
    test_df["price"] - test_df["price_lag_1"]
).abs() > 0.01

baseline_changed_mae = mean_absolute_error(
    test_df.loc[baseline_changed_mask, "price"],
    test_df.loc[baseline_changed_mask, "price_lag_1"]
)

baseline_row = pd.DataFrame([
    {
        "model": "Last Price Baseline",
        "MAE": baseline_classic_metrics["MAE"],
        "RMSE": baseline_classic_metrics["RMSE"],
        "MAPE": baseline_classic_metrics["MAPE"],
        "R2": baseline_classic_metrics["R2"],
        "changed_MAE": baseline_changed_mae,
        "changed_baseline_MAE": baseline_changed_mae
    }
])

classic_split_comparison = pd.concat(
    [
        baseline_row,
        classic_split_results
    ],
    ignore_index=True
).sort_values("MAE")

classic_split_comparison

,model,MAE,RMSE,MAPE,R2,changed_MAE,changed_baseline_MAE
0,Last Price Baseline,2.668074,18.666267,1.449124,0.994025,36.825290,36.82529
1,LightGBM,4.814953,19.411334,3.506027,0.993539,35.737021,36.82529
2,CatBoost,6.706927,29.025122,4.879151,0.985554,36.221466,36.82529
3,XGBoost,7.672987,34.534578,3.851350,0.979549,40.431450,36.82529


In [48]:
all_walk_forward_results = []
all_walk_forward_predictions = []

for model_name, candidate_model in models.items():
    print(f"Çalışıyor: {model_name}")

    model_results, model_predictions = (
        run_walk_forward_validation(
            data=data,
            model=candidate_model,
            model_name=model_name,
            feature_columns=feature_columns,
            folds=walk_forward_folds
        )
    )

    all_walk_forward_results.append(
        model_results
    )

    all_walk_forward_predictions.append(
        model_predictions
    )

walk_forward_model_results = pd.concat(
    all_walk_forward_results,
    ignore_index=True
)

walk_forward_model_predictions = pd.concat(
    all_walk_forward_predictions,
    ignore_index=True
)

walk_forward_model_results

Çalışıyor: XGBoost
Çalışıyor: LightGBM
Çalışıyor: CatBoost


,model,fold,train_end,test_end,train_rows,test_rows,changed_rows,model_MAE,model_RMSE,model_MAPE,model_R2,baseline_MAE,baseline_RMSE,baseline_MAPE,baseline_R2,changed_model_MAE,changed_baseline_MAE,MAE_improvement
0,XGBoost,Fold 1,2026-06-01,2026-06-15,262190,3703,223,6.717856,32.726012,3.863534,0.977939,1.637618,12.116339,1.177737,0.996976,24.953418,27.193274,-5.080238
1,XGBoost,Fold 2,2026-06-15,2026-07-01,265893,4295,206,7.806832,39.483953,3.415174,0.972366,1.814137,13.895255,0.849934,0.996578,41.988141,37.823883,-5.992695
2,XGBoost,Fold 3,2026-07-01,2026-07-18,270188,4516,351,8.083310,37.871196,4.037850,0.973081,3.133459,20.536875,1.691849,0.992084,43.758242,40.315385,-4.949852
3,LightGBM,Fold 1,2026-06-01,2026-06-15,262190,3703,223,3.324683,13.391147,3.530748,0.996306,1.637618,12.116339,1.177737,0.996976,27.377732,27.193274,-1.687065
4,LightGBM,Fold 2,2026-06-15,2026-07-01,265893,4295,206,4.234967,16.213312,2.873302,0.995340,1.814137,13.895255,0.849934,0.996578,38.138840,37.823883,-2.420830
5,LightGBM,Fold 3,2026-07-01,2026-07-18,270188,4516,351,4.694980,20.029074,3.569591,0.992471,3.133459,20.536875,1.691849,0.992084,39.038508,40.315385,-1.561521
6,CatBoost,Fold 1,2026-06-01,2026-06-15,262190,3703,223,5.576539,24.402172,5.218930,0.987734,1.637618,12.116339,1.177737,0.996976,28.743527,27.193274,-3.938921
7,CatBoost,Fold 2,2026-06-15,2026-07-01,265893,4295,206,5.891993,18.769490,4.188456,0.993755,1.814137,13.895255,0.849934,0.996578,40.391984,37.823883,-4.077855
8,CatBoost,Fold 3,2026-07-01,2026-07-18,270188,4516,351,6.094684,22.961785,4.868726,0.990104,3.133459,20.536875,1.691849,0.992084,40.827804,40.315385,-2.961225


In [49]:
walk_forward_model_results[
    [
        "model",
        "fold",
        "train_rows",
        "test_rows",
        "changed_rows",
        "model_MAE",
        "baseline_MAE",
        "MAE_improvement",
        "changed_model_MAE",
        "changed_baseline_MAE",
        "model_R2"
    ]
].sort_values(
    ["fold", "model"]
)

,model,fold,train_rows,test_rows,changed_rows,model_MAE,baseline_MAE,MAE_improvement,changed_model_MAE,changed_baseline_MAE,model_R2
6,CatBoost,Fold 1,262190,3703,223,5.576539,1.637618,-3.938921,28.743527,27.193274,0.987734
3,LightGBM,Fold 1,262190,3703,223,3.324683,1.637618,-1.687065,27.377732,27.193274,0.996306
0,XGBoost,Fold 1,262190,3703,223,6.717856,1.637618,-5.080238,24.953418,27.193274,0.977939
7,CatBoost,Fold 2,265893,4295,206,5.891993,1.814137,-4.077855,40.391984,37.823883,0.993755
4,LightGBM,Fold 2,265893,4295,206,4.234967,1.814137,-2.420830,38.138840,37.823883,0.995340
1,XGBoost,Fold 2,265893,4295,206,7.806832,1.814137,-5.992695,41.988141,37.823883,0.972366
8,CatBoost,Fold 3,270188,4516,351,6.094684,3.133459,-2.961225,40.827804,40.315385,0.990104
5,LightGBM,Fold 3,270188,4516,351,4.694980,3.133459,-1.561521,39.038508,40.315385,0.992471
2,XGBoost,Fold 3,270188,4516,351,8.083310,3.133459,-4.949852,43.758242,40.315385,0.973081


In [50]:
walk_forward_summary = (
    walk_forward_model_results
    .groupby("model")
    .agg(
        mean_MAE=("model_MAE", "mean"),
        std_MAE=("model_MAE", "std"),

        mean_baseline_MAE=("baseline_MAE", "mean"),
        std_baseline_MAE=("baseline_MAE", "std"),

        mean_MAE_improvement=("MAE_improvement", "mean"),
        std_MAE_improvement=("MAE_improvement", "std"),

        mean_changed_MAE=("changed_model_MAE", "mean"),
        std_changed_MAE=("changed_model_MAE", "std"),

        mean_changed_baseline_MAE=(
            "changed_baseline_MAE",
            "mean"
        ),

        mean_RMSE=("model_RMSE", "mean"),
        std_RMSE=("model_RMSE", "std"),

        mean_MAPE=("model_MAPE", "mean"),
        std_MAPE=("model_MAPE", "std"),

        mean_R2=("model_R2", "mean"),
        std_R2=("model_R2", "std")
    )
    .reset_index()
    .sort_values("mean_MAE")
)

walk_forward_summary

,model,mean_MAE,std_MAE,mean_baseline_MAE,std_baseline_MAE,mean_MAE_improvement,std_MAE_improvement,mean_changed_MAE,std_changed_MAE,mean_changed_baseline_MAE,mean_RMSE,std_RMSE,mean_MAPE,std_MAPE,mean_R2,std_R2
1,LightGBM,4.084877,0.697369,2.195071,0.817446,-1.889805,0.464145,34.851694,6.488253,35.110847,16.544511,3.331334,3.324547,0.391272,0.994706,0.001995
0,CatBoost,5.854405,0.261109,2.195071,0.817446,-3.659334,0.608558,36.654439,6.854515,35.110847,22.044482,2.926237,4.758704,0.523973,0.990531,0.003033
2,XGBoost,7.536000,0.721893,2.195071,0.817446,-5.340928,0.568199,36.899933,10.383773,35.110847,36.693720,3.529487,3.772186,0.321231,0.974462,0.003032


In [51]:
baseline_win_summary = (
    walk_forward_model_results
    .assign(
        beats_baseline=lambda df:
            df["model_MAE"] < df["baseline_MAE"]
    )
    .groupby("model")
    .agg(
        total_folds=("fold", "count"),
        folds_beating_baseline=("beats_baseline", "sum"),
        average_improvement=("MAE_improvement", "mean")
    )
    .reset_index()
)

baseline_win_summary["win_rate"] = (
    baseline_win_summary["folds_beating_baseline"]
    / baseline_win_summary["total_folds"]
)

baseline_win_summary.sort_values(
    "win_rate",
    ascending=False
)

,model,total_folds,folds_beating_baseline,average_improvement,win_rate
0,CatBoost,3,0,-3.659334,0.0
1,LightGBM,3,0,-1.889805,0.0
2,XGBoost,3,0,-5.340928,0.0


In [52]:
changed_day_summary = (
    walk_forward_model_results
    .groupby("model")
    .agg(
        changed_rows=("changed_rows", "sum"),
        model_changed_MAE=("changed_model_MAE", "mean"),
        baseline_changed_MAE=(
            "changed_baseline_MAE",
            "mean"
        )
    )
    .reset_index()
)

changed_day_summary["changed_MAE_improvement"] = (
    changed_day_summary["baseline_changed_MAE"]
    - changed_day_summary["model_changed_MAE"]
)

changed_day_summary.sort_values(
    "model_changed_MAE"
)

,model,changed_rows,model_changed_MAE,baseline_changed_MAE,changed_MAE_improvement
1,LightGBM,780,34.851694,35.110847,0.259153
0,CatBoost,780,36.654439,35.110847,-1.543591
2,XGBoost,780,36.899933,35.110847,-1.789086


In [53]:
final_model_comparison = (
    walk_forward_summary[
        [
            "model",
            "mean_MAE",
            "std_MAE",
            "mean_MAE_improvement",
            "mean_changed_MAE",
            "mean_changed_baseline_MAE",
            "mean_RMSE",
            "mean_MAPE",
            "mean_R2"
        ]
    ]
    .merge(
        baseline_win_summary[
            [
                "model",
                "folds_beating_baseline",
                "total_folds",
                "win_rate"
            ]
        ],
        on="model",
        how="left"
    )
)

final_model_comparison["reliable_candidate"] = (
    (final_model_comparison["mean_MAE_improvement"] > 0)
    & (final_model_comparison["win_rate"] >= 2 / 3)
    & (
        final_model_comparison["mean_changed_MAE"]
        <
        final_model_comparison[
            "mean_changed_baseline_MAE"
        ]
    )
)

final_model_comparison.sort_values(
    [
        "reliable_candidate",
        "mean_MAE"
    ],
    ascending=[False, True]
)

,model,mean_MAE,std_MAE,mean_MAE_improvement,mean_changed_MAE,mean_changed_baseline_MAE,mean_RMSE,mean_MAPE,mean_R2,folds_beating_baseline,total_folds,win_rate,reliable_candidate
0,LightGBM,4.084877,0.697369,-1.889805,34.851694,35.110847,16.544511,3.324547,0.994706,0,3,0.0,False
1,CatBoost,5.854405,0.261109,-3.659334,36.654439,35.110847,22.044482,4.758704,0.990531,0,3,0.0,False
2,XGBoost,7.536000,0.721893,-5.340928,36.899933,35.110847,36.693720,3.772186,0.974462,0,3,0.0,False
